In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install networkx

In [ ]:
import subprocess
import sys
import os

def clean_install():
    """Complete clean installation with correct versions"""
    
    print("🧹 Step 1: Cleaning old packages...")
    subprocess.run([
        sys.executable, "-m", "pip", "uninstall", "-y", 
        "torch", "torchvision", "torchaudio", "transformers",
        "numpy", "scikit-learn", "tensorflow", "keras"
    ], capture_output=True)
    
    print("📦 Step 2: Installing core packages...")
    # Install numpy first with specific version
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy==1.24.3"
    ], check=True)
    
    print("🔥 Step 3: Installing PyTorch...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.3.0", "torchvision==0.18.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ], check=True)
    
    print("🤖 Step 4: Installing Transformers...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.44.2"
    ], check=True)
    
    print("📚 Step 5: Installing other dependencies...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers", "faiss-cpu", 
        "spacy", "networkx", "datasets", "rank_bm25"
    ], check=True)
    
    print("🌐 Step 6: Downloading spacy model...")
    subprocess.run([
        sys.executable, "-m", "spacy", "download", "en_core_web_sm"
    ], capture_output=True)
    
    print("\n" + "="*70)
    print("✅ INSTALLATION COMPLETE!")
    print("="*70)
    print("🔴 IMPORTANT: RESTART RUNTIME NOW!")
    print("="*70)
    print("\nGoogle Colab: Runtime → Restart Runtime")
    print("Jupyter: Kernel → Restart Kernel")
    print("\nAfter restart, run the main code below")
    print("="*70)

# Run installation
clean_install()

In [34]:
%%writefile kaggle_persistence.py
import os
import shutil
import pickle
import numpy as np

def save_kg_to_output(kg_db_path="/kaggle/working/kg_database"):
    """Save full KG state to /kaggle/output for Kaggle dataset creation"""
    output_dir = "/kaggle/output/kg_database"
    os.makedirs(output_dir, exist_ok=True)
    files = [
        "documents.pkl", "titles.pkl", "document_embeddings.npy",
        "knowledge_graph.pkl", "entity_contexts.pkl", "entity_cooccurrence.pkl",
        "community_summaries.pkl", "communities.npy", "metadata.json", "query_history.json"
    ]
    saved = 0
    for f in files:
        src = f"{kg_db_path}/{f}"
        if os.path.exists(src):
            shutil.copy2(src, f"{output_dir}/{f}")
            size_mb = os.path.getsize(f"{output_dir}/{f}") / (1024*1024)
            print(f"  ✅ {f} ({size_mb:.2f} MB)")
            saved += 1
    print(f"\n✅ Saved {saved} files to output")
    
def load_kg_from_dataset(dataset_path="/kaggle/input/scientific-kg-dataset2"):
    """Load pre-built KG from Kaggle dataset"""
    working_dir = "/kaggle/working/kg_database"
    os.makedirs(working_dir, exist_ok=True)
    if not os.path.exists(dataset_path):
        print("❌ Dataset path not found")
        return None
    files = os.listdir(dataset_path)
    for f in files:
        shutil.copy2(f"{dataset_path}/{f}", f"{working_dir}/{f}")
    print(f"✅ Loaded KG from dataset")
    return working_dir

Overwriting kaggle_persistence.py


In [32]:
import os



dataset_path = "/kaggle/input/scientific-kg-dataset2"



if os.path.exists(dataset_path):

    print(f"✅ Dataset found at: {dataset_path}")

    print("Files inside:")

    for f in os.listdir(dataset_path):

        print(f"  - {f}")

else:

    print(f"❌ Dataset NOT found at: {dataset_path}")

    print("➡️ Go to 'Datasets' panel → find your dataset → click 'Add to notebook'")

✅ Dataset found at: /kaggle/input/scientific-kg-dataset2
Files inside:
  - query_history.json
  - knowledge_graph.pkl
  - documents.pkl
  - metadata.json
  - document_embeddings.npy
  - titles.pkl
  - communities.npy
  - entity_contexts.pkl
  - entity_cooccurrence.pkl
  - community_summaries.pkl
  - knowledge_graph_export.json


In [41]:

    dataset_path="/kaggle/input/scientific-kg-dataset2"
    working_dir = "/kaggle/working/kg_database"
    os.makedirs(working_dir, exist_ok=True)
    if not os.path.exists(dataset_path):
        print("❌ Dataset path not found")
        
    files = os.listdir(dataset_path)
    for f in files:
        shutil.copy2(f"{dataset_path}/{f}", f"{working_dir}/{f}")
    print(f"✅ Loaded KG from dataset")

    kg_path=working_dir
    

✅ Loaded KG from dataset


In [12]:
from kaggle_persistence import load_kg_from_dataset

# Load directly from the dataset root
kg_path = load_kg_from_dataset("/kaggle/input/scientific-kg-dataset2")

if kg_path:
    print(f"✅ KG loaded successfully at: {kg_path}")
else:
    print("❌ Failed to load KG")

✅ Loaded KG from dataset
✅ KG loaded successfully at: /kaggle/working/kg_database


In [42]:
#-------------------------------------------------------#
import torch
import faiss
import numpy as np
import spacy
import time
import networkx as nx
import json
import pickle
import os
from datetime import datetime
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from collections import defaultdict

class HIERARCHICAL_LIGHT_RAG:
    def __init__(self, kg_db_path="kg_database"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🚀 Device: {self.device}")
        # KG Database setup
        self.kg_db_path = f"/kaggle/working/{kg_db_path}"
        os.makedirs(self.kg_db_path, exist_ok=True)
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except Exception as e:
            print(f"⚠️ spaCy loading failed: {e}. Falling back to basic processing.")
            self.nlp = None
        print("🔄 Loading embedding model...")
        self.embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        print("🔄 Loading generator model...")
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.generator = AutoModelForSeq2SeqLM.from_pretrained(
            "google/flan-t5-base",
            torch_dtype=dtype,
            device_map="auto" if self.device == "cuda" else None,
            low_cpu_mem_usage=True
        )
        if self.device == "cpu":
            self.generator = self.generator.to(self.device)
        # Core data structures
        self.documents = []
        self.titles = []
        self.document_embeddings = None
        # LIGHTRAG: Hybrid retrieval components
        self.neighbor_index = None
        self.community_index = None
        self.communities = []
        self.community_summaries = {}
        self.community_graph = nx.Graph()
        # Dynamic Knowledge Graph with Local DB
        self.kg = nx.Graph()
        self.entity_contexts = defaultdict(list)
        self.entity_cooccurrence = defaultdict(int)
        self.query_history = []
        # BM25 for sparse retrieval
        self.bm25 = None
        # Load existing KG if available
        self.load_kg_database()
        print("✅ Models loaded and Hybrid RAG initialized")

    def _full_state_exists(self):
        required_files = [
            "documents.pkl", "titles.pkl", "document_embeddings.npy",
            "knowledge_graph.pkl", "entity_contexts.pkl", "entity_cooccurrence.pkl",
            "community_summaries.pkl", "communities.npy", "metadata.json"
        ]
        return all(os.path.exists(f"{self.kg_db_path}/{f}") for f in required_files)

    def save_full_state(self):
        """Save full state for fast reload and Kaggle persistence"""
        print("💾 Saving full state...")
        os.makedirs(self.kg_db_path, exist_ok=True)
        with open(f"{self.kg_db_path}/documents.pkl", "wb") as f:
            pickle.dump(self.documents, f)
        with open(f"{self.kg_db_path}/titles.pkl", "wb") as f:
            pickle.dump(self.titles, f)
        np.save(f"{self.kg_db_path}/document_embeddings.npy", self.document_embeddings)
        with open(f"{self.kg_db_path}/knowledge_graph.pkl", "wb") as f:
            pickle.dump(self.kg, f)
        with open(f"{self.kg_db_path}/entity_contexts.pkl", "wb") as f:
            pickle.dump(dict(self.entity_contexts), f)
        with open(f"{self.kg_db_path}/entity_cooccurrence.pkl", "wb") as f:
            pickle.dump(dict(self.entity_cooccurrence), f)
        with open(f"{self.kg_db_path}/community_summaries.pkl", "wb") as f:
            pickle.dump(self.community_summaries, f)
        with open(f"{self.kg_db_path}/communities.npy", "wb") as f:
            np.save(f, np.array(self.communities))
        with open(f"{self.kg_db_path}/query_history.json", "w") as f:
            json.dump(self.query_history[-100:], f, indent=2)
        metadata = {
            'last_updated': datetime.now().isoformat(),
            'total_entities': self.kg.number_of_nodes(),
            'total_relationships': self.kg.number_of_edges(),
            'total_queries': len(self.query_history),
            'total_documents': len(self.documents)
        }
        with open(f"{self.kg_db_path}/metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
        print(f"✅ Full state saved: {metadata['total_entities']} entities, {metadata['total_relationships']} relationships")

    def load_full_state(self):
        """Reload full state from disk quickly"""
        print("📂 Loading full state...")
        with open(f"{self.kg_db_path}/documents.pkl", "rb") as f:
            self.documents = pickle.load(f)
        with open(f"{self.kg_db_path}/titles.pkl", "rb") as f:
            self.titles = pickle.load(f)
        self.document_embeddings = np.load(f"{self.kg_db_path}/document_embeddings.npy")
        with open(f"{self.kg_db_path}/knowledge_graph.pkl", "rb") as f:
            self.kg = pickle.load(f)
        with open(f"{self.kg_db_path}/entity_contexts.pkl", "rb") as f:
            self.entity_contexts = defaultdict(list, pickle.load(f))
        with open(f"{self.kg_db_path}/entity_cooccurrence.pkl", "rb") as f:
            self.entity_cooccurrence = defaultdict(int, pickle.load(f))
        with open(f"{self.kg_db_path}/community_summaries.pkl", "rb") as f:
            self.community_summaries = pickle.load(f)
        self.communities = np.load(f"{self.kg_db_path}/communities.npy").tolist()
        if os.path.exists(f"{self.kg_db_path}/query_history.json"):
            with open(f"{self.kg_db_path}/query_history.json", "r") as f:
                self.query_history = json.load(f)
        # Rebuild FAISS index
        dim = self.document_embeddings.shape[1]
        self.neighbor_index = faiss.IndexFlatIP(dim)
        emb_norm = self.document_embeddings.astype(np.float32)
        faiss.normalize_L2(emb_norm)
        self.neighbor_index.add(emb_norm)
        # Rebuild BM25
        tokenized_docs = [doc.lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        # Rebuild community index
        centroids = []
        for cid in set(self.communities):
            mask = np.array(self.communities) == cid
            if np.any(mask):
                centroids.append(np.mean(self.document_embeddings[mask], axis=0))
        self.community_index = faiss.IndexFlatIP(dim)
        centroids = np.array(centroids).astype(np.float32)
        faiss.normalize_L2(centroids)
        self.community_index.add(centroids)
        print("✅ Full state reloaded")

    def save_kg_database(self):
        """Save KG and related data to local database"""
        print("💾 Saving KG database...")
        try:
            # Save KG graph
            with open(f"{self.kg_db_path}/knowledge_graph.pkl", 'wb') as f:
                pickle.dump(self.kg, f)
            # Save entity contexts
            with open(f"{self.kg_db_path}/entity_contexts.pkl", 'wb') as f:
                pickle.dump(dict(self.entity_contexts), f)
            # Save entity co-occurrence
            with open(f"{self.kg_db_path}/entity_cooccurrence.pkl", 'wb') as f:
                pickle.dump(dict(self.entity_cooccurrence), f)
            # Save query history as JSON for readability
            with open(f"{self.kg_db_path}/query_history.json", 'w') as f:
                json.dump(self.query_history[-100:], f, indent=2)  # Keep last 100 queries
            # Save community summaries
            with open(f"{self.kg_db_path}/community_summaries.pkl", 'wb') as f:
                pickle.dump(self.community_summaries, f)
            # Save metadata
            metadata = {
                'last_updated': datetime.now().isoformat(),
                'total_entities': self.kg.number_of_nodes(),
                'total_relationships': self.kg.number_of_edges(),
                'total_queries': len(self.query_history),
                'total_documents': len(self.documents)
            }
            with open(f"{self.kg_db_path}/metadata.json", 'w') as f:
                json.dump(metadata, f, indent=2)
            print(f"   ✅ KG Database saved: {metadata['total_entities']} entities, {metadata['total_relationships']} relationships")
        except Exception as e:
            print(f"   ❌ Error saving KG database: {e}")

    def load_kg_database(self):
        """Load KG and related data from local database"""
        print("📂 Loading KG database...")
        try:
            # Load KG graph
            kg_path = f"{self.kg_db_path}/knowledge_graph.pkl"
            if os.path.exists(kg_path):
                with open(kg_path, 'rb') as f:
                    self.kg = pickle.load(f)
                print(f"   ✅ Loaded KG: {self.kg.number_of_nodes()} entities, {self.kg.number_of_edges()} relationships")
            # Load entity contexts
            contexts_path = f"{self.kg_db_path}/entity_contexts.pkl"
            if os.path.exists(contexts_path):
                with open(contexts_path, 'rb') as f:
                    self.entity_contexts.update(pickle.load(f))
                print(f"   ✅ Loaded {len(self.entity_contexts)} entity contexts")
            # Load entity co-occurrence
            cooccur_path = f"{self.kg_db_path}/entity_cooccurrence.pkl"
            if os.path.exists(cooccur_path):
                with open(cooccur_path, 'rb') as f:
                    self.entity_cooccurrence.update(pickle.load(f))
                print(f"   ✅ Loaded {len(self.entity_cooccurrence)} co-occurrence records")
            # Load query history
            history_path = f"{self.kg_db_path}/query_history.json"
            if os.path.exists(history_path):
                with open(history_path, 'r') as f:
                    self.query_history = json.load(f)
                print(f"   ✅ Loaded {len(self.query_history)} query history entries")
            # Load community summaries
            community_path = f"{self.kg_db_path}/community_summaries.pkl"
            if os.path.exists(community_path):
                with open(community_path, 'rb') as f:
                    self.community_summaries = pickle.load(f)
                print(f"   ✅ Loaded {len(self.community_summaries)} community summaries")
            # Load metadata
            metadata_path = f"{self.kg_db_path}/metadata.json"
            if os.path.exists(metadata_path):
                with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                print(f"   📊 Database stats: {metadata}")
        except Exception as e:
            print(f"   ❌ Error loading KG database: {e}")

    def export_kg_to_json(self, filename="knowledge_graph_export.json"):
        """Export KG to human-readable JSON format"""
        export_data = {
            'entities': {},
            'relationships': [],
            'metadata': {
                'export_date': datetime.now().isoformat(),
                'total_entities': self.kg.number_of_nodes(),
                'total_relationships': self.kg.number_of_edges(),
                'total_queries': len(self.query_history)
            }
        }
        # Export entities with their contexts
        for entity in self.kg.nodes():
            export_data['entities'][entity] = {
                'contexts': self.entity_contexts.get(entity, [])[:5],  # Top 5 contexts
                'degree_centrality': self.kg.degree(entity),
                'neighbors': list(self.kg.neighbors(entity))[:10]  # Top 10 neighbors
            }
        # Export relationships
        for edge in self.kg.edges(data=True):
            export_data['relationships'].append({
                'source': edge[0],
                'target': edge[1],
                'weight': edge[2].get('weight', 1),
                'strength': 'strong' if edge[2].get('weight', 1) > 2 else 'weak'
            })
        # Save to file
        with open(f"{self.kg_db_path}/{filename}", 'w') as f:
            json.dump(export_data, f, indent=2, ensure_ascii=False)
        print(f"📤 KG exported to {filename}")
        return export_data

    def get_kg_statistics(self):
        """Get comprehensive statistics about the knowledge graph"""
        if self.kg.number_of_nodes() == 0:
            return {"message": "Knowledge graph is empty"}
        # Basic stats
        stats = {
            'total_entities': self.kg.number_of_nodes(),
            'total_relationships': self.kg.number_of_edges(),
            'density': nx.density(self.kg),
            'average_degree': sum(dict(self.kg.degree()).values()) / self.kg.number_of_nodes(),
            'connected_components': nx.number_connected_components(self.kg),
            'total_queries': len(self.query_history),
            'total_entity_contexts': sum(len(contexts) for contexts in self.entity_contexts.values())
        }
        # Top entities by degree centrality
        degree_centrality = nx.degree_centrality(self.kg)
        top_entities = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
        stats['top_entities'] = top_entities
        # Strongest relationships
        edges_with_weights = [(u, v, d['weight']) for u, v, d in self.kg.edges(data=True)]
        top_relationships = sorted(edges_with_weights, key=lambda x: x[2], reverse=True)[:10]
        stats['top_relationships'] = top_relationships
        return stats

    def extract_entities(self, text):
        """Extract entities using spaCy or fallback method"""
        entities = set()
        if self.nlp:
            doc = self.nlp(text)
            for ent in doc.ents:
                if ent.label_ in ("PERSON", "GPE", "ORG", "LOC", "NORP", "EVENT", "WORK_OF_ART"):
                    entities.add(ent.text.lower())
            for token in doc:
                if token.pos_ in ("PROPN", "NOUN") and len(token.text) > 3 and token.is_alpha:
                    entities.add(token.text.lower())
        else:
            words = text.lower().split()
            for word in words:
                if len(word) > 3 and word.isalpha():
                    entities.add(word)
        return entities

    def build_knowledge_graph(self, query, retrieved_docs):
        """Build KG dynamically from query and retrieved documents"""
        print("🔄 Building dynamic knowledge graph...")
        # Add to query history
        self.query_history.append({
            'query': query,
            'timestamp': datetime.now().isoformat(),
            'retrieved_docs': len(retrieved_docs)
        })
        # Extract entities from query and documents
        all_entities = set()
        query_entities = self.extract_entities(query)
        all_entities.update(query_entities)
        doc_entities_list = []
        for doc in retrieved_docs[:5]:
            doc_entities = self.extract_entities(doc['content'][:1000])
            all_entities.update(doc_entities)
            doc_entities_list.append(doc_entities)
            # Add document context to entities
            for entity in doc_entities:
                self.entity_contexts[entity].append({
                    'title': doc['title'],
                    'snippet': doc['content'][:200],
                    'source': 'document',
                    'timestamp': datetime.now().isoformat()
                })
        # Add query context
        for entity in query_entities:
            self.entity_contexts[entity].append({
                'title': 'User Query',
                'snippet': query,
                'source': 'query',
                'timestamp': datetime.now().isoformat()
            })
        # Build co-occurrence relationships
        for entities in doc_entities_list:
            entities_list = list(entities)
            for i, ent1 in enumerate(entities_list):
                for ent2 in entities_list[i+1:]:
                    if ent1 != ent2:
                        if self.kg.has_edge(ent1, ent2):
                            self.kg[ent1][ent2]['weight'] += 1
                        else:
                            self.kg.add_edge(ent1, ent2, weight=1)
                        self.entity_cooccurrence[(ent1, ent2)] += 1
        # Add query entity relationships
        query_entities_list = list(query_entities)
        for i, ent1 in enumerate(query_entities_list):
            for ent2 in query_entities_list[i+1:]:
                if ent1 != ent2:
                    if self.kg.has_edge(ent1, ent2):
                        self.kg[ent1][ent2]['weight'] += 2
                    else:
                        self.kg.add_edge(ent1, ent2, weight=2)
                    self.entity_cooccurrence[(ent1, ent2)] += 2
        print(f"   ✅ KG now has {self.kg.number_of_nodes()} entities and {self.kg.number_of_edges()} relationships")
        # Save to database
        self.save_kg_database()

    def find_kg_paths(self, query, max_paths=5, max_depth=2):
        """Find reasoning paths in the knowledge graph"""
        query_entities = self.extract_entities(query)
        all_paths = []
        for entity in query_entities:
            if entity in self.kg:
                for target_entity in self.kg.nodes():
                    if entity != target_entity:
                        try:
                            path = nx.shortest_path(self.kg, source=entity, target=target_entity)
                            if 2 <= len(path) <= max_depth + 1:
                                all_paths.append(path)
                        except nx.NetworkXNoPath:
                            continue
        # Sort paths by strength
        scored_paths = []
        for path in all_paths:
            strength = 0
            for i in range(len(path)-1):
                strength += self.kg[path[i]][path[i+1]]['weight']
            scored_paths.append((path, strength))
        scored_paths.sort(key=lambda x: x[1], reverse=True)
        return [path for path, strength in scored_paths[:max_paths]]

    def get_entity_context(self, entity, max_contexts=3):
        """Get relevant context for an entity"""
        if entity in self.entity_contexts:
            contexts = self.entity_contexts[entity][:max_contexts]
            return [f"{ctx['title']}: {ctx['snippet']}" for ctx in contexts]
        return []

    def build_lightrag_communities(self, n_clusters=50):
        """LIGHTRAG: Build document communities using clustering"""
        print("🔄 Building LIGHTRAG communities...")
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.communities = kmeans.fit_predict(self.document_embeddings)
        community_centroids = []
        for i in range(n_clusters):
            cluster_mask = (self.communities == i)
            if np.sum(cluster_mask) > 0:
                centroid = np.mean(self.document_embeddings[cluster_mask], axis=0)
                community_centroids.append(centroid)
        dim = self.document_embeddings.shape[1]
        self.community_index = faiss.IndexFlatIP(dim)
        community_centroids = np.array(community_centroids).astype(np.float32)
        faiss.normalize_L2(community_centroids)
        self.community_index.add(community_centroids)
        print(f"✅ Built {n_clusters} communities")

    def generate_community_summaries(self):
        """RAPTOR: Generate hierarchical summaries for each community"""
        print("🔄 Generating RAPTOR-style community summaries...")
        for community_id in set(self.communities):
            community_docs = [self.documents[i] for i in range(len(self.documents)) 
                            if self.communities[i] == community_id]
            if len(community_docs) > 0:
                community_indices = [i for i in range(len(self.documents)) 
                                   if self.communities[i] == community_id]
                community_embeddings = self.document_embeddings[community_indices]
                centroid = np.mean(community_embeddings, axis=0)
                similarities = np.dot(community_embeddings, centroid)
                top_indices = np.argsort(similarities)[-3:][::-1]
                summary_docs = [community_docs[i] for i in top_indices]
                summary_texts = [doc[:500] for doc in summary_docs[:2]]
                prompt = f"Summarize the main themes and key information from these related documents:\n" + "\n".join(summary_texts)
                inputs = self.tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                with torch.no_grad():
                    outputs = self.generator.generate(
                        **inputs, max_length=200, num_beams=4, 
                        early_stopping=True, temperature=0.7
                    )
                summary = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                self.community_summaries[community_id] = summary
        print(f"✅ Generated {len(self.community_summaries)} community summaries")
        # Save community summaries to database
        self.save_kg_database()

    def lightrag_hybrid_retrieve(self, query, k=8):
        """LIGHTRAG CORE: Hybrid retrieval (neighbors + communities)"""
        query_embedding = self.embedder.encode([query])
        faiss.normalize_L2(query_embedding)
        neighbor_scores, neighbor_indices = self.neighbor_index.search(
            query_embedding.astype(np.float32), k * 2
        )
        community_scores, community_ids = self.community_index.search(
            query_embedding.astype(np.float32), 5
        )
        community_doc_indices = []
        for community_id in community_ids[0]:
            community_docs = [i for i in range(len(self.documents)) 
                            if self.communities[i] == community_id]
            community_doc_indices.extend(community_docs[:2])
        all_indices = list(neighbor_indices[0]) + community_doc_indices
        unique_indices = list(dict.fromkeys(all_indices))
        relevant_communities = set()
        for idx in unique_indices[:k]:
            relevant_communities.add(self.communities[idx])
        community_context = []
        for comm_id in relevant_communities:
            if comm_id in self.community_summaries:
                community_context.append(f"Community Theme: {self.community_summaries[comm_id]}")
        return unique_indices[:k], community_context

    def load_documents(self, num_docs=3000):
        print(f"📥 Loading {num_docs} documents...")
        dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
        cnt = 0
        for item in dataset:
            if cnt >= num_docs:
                break
            if 'text' not in item or 'title' not in item: 
                continue
            title, text = item['title'], item['text']
            if len(text) < 300: 
                continue
            self.documents.append(text)
            self.titles.append(title)
            cnt += 1
            if cnt % 1000 == 0:
                print(f"   📖 {cnt} documents")
        print(f"✅ Loaded {len(self.documents)} documents")
        self._build_hybrid_indices()

    def _build_hybrid_indices(self):
        """Build both LIGHTRAG and RAPTOR indices"""
        print("🔍 Building hybrid search indices...")
        texts = [f"{t}: {d[:600]}" for t, d in zip(self.titles, self.documents)]
        self.document_embeddings = self.embedder.encode(texts, show_progress_bar=True)
        dim = self.document_embeddings.shape[1]
        self.neighbor_index = faiss.IndexFlatIP(dim)
        embeddings_normalized = self.document_embeddings.astype(np.float32)
        faiss.normalize_L2(embeddings_normalized)
        self.neighbor_index.add(embeddings_normalized)
        self.build_lightrag_communities(n_clusters=min(50, len(self.documents)//10))
        self.generate_community_summaries()
        print("   Building BM25 index...")
        tokenized_docs = [doc.lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        print(f"✅ Hybrid indices built with {self.neighbor_index.ntotal} vectors")

    def retrieve(self, query, k=8):
        """Enhanced hybrid retrieval combining LIGHTRAG + RAPTOR"""
        doc_indices, community_context = self.lightrag_hybrid_retrieve(query, k)
        retrieved_docs = [{
            'content': self.documents[idx],
            'title': self.titles[idx],
            'community': self.communities[idx]
        } for idx in doc_indices]
        self.build_knowledge_graph(query, retrieved_docs)
        reasoning_paths = self.find_kg_paths(query)
        enhanced_docs = []
        for idx in doc_indices:
            enhanced_docs.append({
                'content': self.documents[idx],
                'title': self.titles[idx],
                'score': 1.0,
                'community': self.communities[idx]
            })
        return enhanced_docs, community_context, reasoning_paths

    def generate(self, query, docs, community_context, reasoning_paths):
        """Enhanced generation with hybrid context"""
        if not docs:
            return "I couldn't find relevant information to answer this question."
        context_parts = []
        # 1. Document context
        for i, doc in enumerate(docs[:4], 1):
            content = doc['content'][:800].replace('\n', ' ').strip()
            context_parts.append(f"Document {i} ({doc['title']}): {content}")
        # 2. LIGHTRAG community context
        if community_context:
            context_parts.append("THEMATIC CONTEXT:")
            context_parts.extend(community_context[:2])
        # 3. RAPTOR reasoning paths
        if reasoning_paths:
            context_parts.append("KNOWLEDGE PATHS:")
            for i, path in enumerate(reasoning_paths[:3], 1):
                path_str = " → ".join(path)
                context_parts.append(f"Path {i}: {path_str}")
                for entity in path[:2]:
                    entity_contexts = self.get_entity_context(entity, 1)
                    if entity_contexts:
                        context_parts.append(f"   {entity}: {entity_contexts[0][:100]}...")
        prompt = (f"You are a reasoning assistant. Answer the question using the following context.\n"
                 f"CONTEXT:\n{chr(10).join(context_parts)}\n"
                 f"Question: {query}\n"
                 f"Provide a comprehensive answer that synthesizes information from documents, thematic context, and knowledge paths:")
        inputs = self.tokenizer(prompt, return_tensors="pt", max_length=2048, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.generator.generate(
                **inputs, max_length=400, min_length=100, num_beams=6, length_penalty=1.5,
                early_stopping=True, no_repeat_ngram_size=3, temperature=0.8, do_sample=False
            )
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        return answer

    def query(self, question, k=8):
        start = time.time()
        print(f"\n❓ {question}")
        docs, community_context, reasoning_paths = self.retrieve(question, k=k)
        print(f"   📚 Retrieved {len(docs)} documents from {len(set([d['community'] for d in docs]))} communities")
        print(f"   🏷️  Community summaries: {len(community_context)}")
        print(f"   🦅 Reasoning paths: {len(reasoning_paths)}")
        for i, doc in enumerate(docs[:3], 1):
            title_preview = doc['title'][:50] + "..." if len(doc['title']) > 50 else doc['title']
            print(f"      {i}. {title_preview} (community: {doc['community']})")
        if community_context:
            print(f"   🎯 Key themes: {', '.join([ctx[:60] + '...' for ctx in community_context[:2]])}")
        if reasoning_paths:
            print(f"   🛣️  Top path: {' → '.join(reasoning_paths[0][:3])}..." if reasoning_paths[0] else "No paths")
        answer = self.generate(question, docs, community_context, reasoning_paths)
        elapsed = time.time() - start
        print(f"   ✅ ANSWER: {answer}")
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print("-" * 70)
        return {
            'question': question,
            'answer': answer,
            'sources': [d['title'] for d in docs[:4]],
            'community_context': community_context,
            'reasoning_paths': reasoning_paths,
            'kg_stats': {
                'entities': self.kg.number_of_nodes(),
                'relationships': self.kg.number_of_edges()
            },
            'time': elapsed,
            'answer_length': len(answer)
        }

    def show_kg_stats(self):
        """Display comprehensive KG statistics"""
        stats = self.get_kg_statistics()
        print("\n📊 KNOWLEDGE GRAPH STATISTICS")
        print("=" * 50)
        for key, value in stats.items():
            if key not in ['top_entities', 'top_relationships']:
                print(f"   {key.replace('_', ' ').title()}: {value}")
        print(f"\n🏆 Top Entities by Centrality:")
        for entity, centrality in stats.get('top_entities', [])[:5]:
            print(f"   • {entity}: {centrality:.4f}")
        print(f"\n🔗 Strongest Relationships:")
        for source, target, weight in stats.get('top_relationships', [])[:5]:
            print(f"   • {source} ↔ {target} (weight: {weight})")

In [43]:
def main():

    import numpy as np

    import random

    np.random.seed(42)

    random.seed(42)

    

    test_queries = [

        "What is artificial intelligence?",

        "Who was Albert Einstein?",

        "What is the capital of France?",

        "How does photosynthesis work?",

        "Explain machine learning"

    ]

    

    # Initialize with KG database

    rag = HIERARCHICAL_LIGHT_RAG(kg_db_path="kg_database")

    rag.load_documents(num_docs=2000)

    

    print("="*70)

    print("🧪 TESTING HIERARCHICAL-LIGHT RAG WITH KG LOCAL DB")

    print("="*70)

    

    results = []

    for q in test_queries:

        result = rag.query(q, k=6)

        results.append(result)

    

    # Show final statistics

    rag.show_kg_stats()

    

    # Export KG to JSON

    rag.export_kg_to_json("knowledge_graph_export.json")

    

    print(f"\n✨ Final KG Statistics: {rag.kg.number_of_nodes()} entities, {rag.kg.number_of_edges()} relationships")

    print("✨ Hybrid RAG system with Local KG DB ready!")

    return rag, results



if __name__ == "__main__":

    rag, results = main()

🚀 Device: cuda
🔄 Loading embedding model...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


🔄 Loading generator model...
📂 Loading KG database...
   ✅ Loaded KG: 963 entities, 26354 relationships
   ✅ Loaded 964 entity contexts
   ✅ Loaded 26362 co-occurrence records
   ✅ Loaded 12 query history entries
   ✅ Loaded 50 community summaries
   📊 Database stats: {'last_updated': '2025-12-02T19:26:35.770506', 'total_entities': 963, 'total_relationships': 26354, 'total_queries': 12, 'total_documents': 2000}
✅ Models loaded and Hybrid RAG initialized
📥 Loading 2000 documents...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: e0f7c1e1-9ec2-40c4-af51-50a47d1373ed)')' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 1s [Retry 1/5].


...

🔄 Building LIGHTRAG communities...
✅ Built 50 communities
🔄 Generating RAPTOR-style community summaries...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


✅ Generated 50 community summaries
💾 Saving KG database...
   ✅ KG Database saved: 963 entities, 26354 relationships
   Building BM25 index...
✅ Hybrid indices built with 2000 vectors
🧪 TESTING HIERARCHICAL-LIGHT RAG WITH KG LOCAL DB

❓ What is artificial intelligence?
🔄 Building dynamic knowledge graph...
   ✅ KG now has 963 entities and 26354 relationships
💾 Saving KG database...
   ✅ KG Database saved: 963 entities, 26354 relationships
   📚 Retrieved 6 documents from 2 communities
   🏷️  Community summaries: 2
   🦅 Reasoning paths: 5
      1. Artificial intelligence (community: 34)
      2. Ai (community: 34)
      3. AI-complete (community: 34)
   🎯 Key themes: Community Theme: Ada is a high-level programming language...., Community Theme: Steve Wozniak designed and manufactured the...
   🛣️  Top path: intelligence → computer → decision...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   ✅ ANSWER: The intelligence of machines or software, as opposed to the intelligence of humans or animals. It is also the field of study in computer science that develops and studies intelligent machines. "AI" may also refer to the machines themselves. AI technology is widely used throughout industry, government and science. Some high-profile applications are: advanced web search engines (e.g., Google Search), recommendation systems (used by YouTube, Amazon, and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars
   ⏱️  Time: 8.54s
----------------------------------------------------------------------

❓ Who was Albert Einstein?
🔄 Building dynamic knowledge graph...
   ✅ KG now has 963 entities and 26354 relationships
💾 Saving KG database...
   ✅ KG Database saved: 963 entities, 26354 relationships
   📚 Retrieved 6 documents from 3 communities
   🏷️  Community summaries: 3
   🦅 Reasoning paths: 5
      1. Albert Einstein (community: 22)
      2. Albertus M

In [ ]:
#Offline Query Run#

rag.query("What is quantum entanglement?")


❓ What is quantum entanglement?
🔄 Building dynamic knowledge graph...
   ✅ KG now has 963 entities and 26354 relationships
💾 Saving KG database...
   ✅ KG Database saved: 963 entities, 26354 relationships
   📚 Retrieved 8 documents from 6 communities
   🏷️  Community summaries: 6
   🦅 Reasoning paths: 0
      1. BQP (community: 28)
      2. Atomic physics (community: 48)
      3. Bundle theory (community: 20)
   🎯 Key themes: Community Theme: Ada is a high-level programming language...., Community Theme: Aberration, or abbies, refers to sound, as ...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   ✅ ANSWER: a state of matter that is typically formed when a gas of bosons at very low densities is cooled to temperatures very close to absolute zero (273.15 °C or 459.67 °F). This state was first predicted, generally, in 1924–1925 by Albert Einstein, crediting a pioneering paper by Satyendra Nath Bose on the new field now known as quantum statistics. The term atomic physics can be associated with nuclear power and nuclear weapons, due to the synonymous use of atomic and nuclear in standard English.
   ⏱️  Time: 8.94s
----------------------------------------------------------------------


{'question': 'What is quantum entanglement?',
 'answer': 'a state of matter that is typically formed when a gas of bosons at very low densities is cooled to temperatures very close to absolute zero (273.15 °C or 459.67 °F). This state was first predicted, generally, in 1924–1925 by Albert Einstein, crediting a pioneering paper by Satyendra Nath Bose on the new field now known as quantum statistics. The term atomic physics can be associated with nuclear power and nuclear weapons, due to the synonymous use of atomic and nuclear in standard English.',
 'sources': ['BQP',
  'Atomic physics',
  'Bundle theory',
  'Bose–Einstein condensate'],
 'community_context': ['Community Theme: Ada is a high-level programming language.',
  'Community Theme: Aberration, or abbies, refers to sound, as it is transmitted in signal form.',
  'Community Theme: Antimatter is matter composed of the antiparticles (or "partners") of the corresponding particles in "ordinary" matter, and can be thought of as matter

In [16]:
# Setup model dirs
MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save embedding model if new session
if not os.path.exists(f"{MODEL_DIR}/all-mpnet-base-v2"):
    print("📥 Saving embedding model...")
    SentenceTransformer('sentence-transformers/all-mpnet-base-v2').save(f"{MODEL_DIR}/all-mpnet-base-v2")

📥 Saving embedding model...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [21]:
# Save generator model if new session
if not os.path.exists(f"{MODEL_DIR}/flan-t5-base"):
    print("📥 Saving generator model...")
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    tokenizer.save_pretrained(MODEL_DIR + "/flan-t5-base")
    model.save_pretrained(MODEL_DIR + "/flan-t5-base")

📥 Saving generator model...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [44]:
import torch

import faiss

import numpy as np

import spacy

import os

import pickle

from datetime import datetime

from sklearn.cluster import KMeans

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from sentence_transformers import SentenceTransformer

from datasets import load_dataset

from collections import defaultdict

import networkx as nx

import re

import hashlib



"""

INCREMENTAL KG LOADER - Small Batch Updates (500 articles/run)

================================================================

Efficiently expand your Knowledge Graph in small increments:

- Each run: 500 new articles (~200-400 chunks typically)

- Fast execution: ~5-10 minutes per run

- Automatic deduplication via content hashing

- Preserves all existing data and relationships

- Memory efficient for Kaggle's 16GB RAM limit



USAGE:

------

First run:  500 articles → Build initial KG

Second run: 500 more    → Total 1000 articles

Third run:  500 more    → Total 1500 articles

... continue as needed ...

"""



MODEL_DIR = "/kaggle/working/models"

os.makedirs(MODEL_DIR, exist_ok=True)



def chunk_wikipedia_article(title, text, chunk_size=300):

    """Fast chunking with word-based splitting"""

    paragraphs = [p for p in re.split(r'\n+', text) if len(p.strip()) > 0]

    chunks = []

    current = ""

    curr_len = 0

    

    for para in paragraphs:

        para_words = para.strip().split()

        if curr_len + len(para_words) > chunk_size and current:

            chunks.append(current.strip())

            current = ""

            curr_len = 0

        current += " " + para

        curr_len += len(para_words)

    

    if current:

        chunks.append(current.strip())

    

    # Annotate with title for context

    annotated_chunks = [

        f"{title}. {title}. {chunk}"[:1500] 

        for chunk in chunks 

        if len(chunk.split()) > 40

    ]

    return annotated_chunks



def compute_content_hash(text):

    """Generate unique hash for deduplication"""

    return hashlib.md5(text.encode('utf-8')).hexdigest()



class IncrementalKGLoader:

    def __init__(self, kg_db_path="scientific_kg_fast"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.kg_db_path = f"/kaggle/working/{kg_db_path}"

        os.makedirs(self.kg_db_path, exist_ok=True)

        

        # Metadata tracking

        self.metadata_path = f"{self.kg_db_path}/metadata.pkl"

        self.metadata = self._load_metadata()

        

        print(f"🚀 Incremental KG Loader initialized on {self.device}")

        print(f"📊 Current state:")

        print(f"   Articles processed: {self.metadata['total_articles']}")

        print(f"   Chunks stored: {self.metadata['total_chunks']}")

        print(f"   Last update: {self.metadata['last_update'] or 'Never'}")

        

        # Load models (cached after first run)

        print("⚡ Loading models...")

        try:

            self.nlp = spacy.load("en_core_web_sm")

        except:

            print("⚠️ spaCy unavailable, using fallback entity extraction")

            self.nlp = None

        

        self.embedder = SentenceTransformer(f'{MODEL_DIR}/all-mpnet-base-v2')

        self.stop_words = set(['the','of','a','an','and','or','but','in','on','to','with','for','by','from','that','this','is','are','it'])

        print("✅ Models loaded\n")



    def _load_metadata(self):

        """Load or initialize metadata"""

        if os.path.exists(self.metadata_path):

            with open(self.metadata_path, "rb") as f:

                return pickle.load(f)

        else:

            return {

                'total_articles': 0,

                'total_chunks': 0,

                'content_hashes': set(),

                'last_update': None,

                'update_history': []

            }

    

    def _save_metadata(self):

        """Save metadata"""

        with open(self.metadata_path, "wb") as f:

            pickle.dump(self.metadata, f)



    def load_existing_kg(self):

        """Load existing KG or create new one"""

        docs_file = f"{self.kg_db_path}/documents.pkl"

        

        if os.path.exists(docs_file):

            print("📂 Loading existing KG...")

            

            with open(docs_file, "rb") as f:

                documents = pickle.load(f)

            with open(f"{self.kg_db_path}/titles.pkl", "rb") as f:

                titles = pickle.load(f)

            embeddings = np.load(f"{self.kg_db_path}/document_embeddings.npy")

            

            kg_file = f"{self.kg_db_path}/knowledge_graph.pkl"

            if os.path.exists(kg_file):

                with open(kg_file, "rb") as f:

                    kg = pickle.load(f)

                with open(f"{self.kg_db_path}/entity_to_doc.pkl", "rb") as f:

                    entity_to_doc = defaultdict(list, pickle.load(f))

            else:

                kg = nx.Graph()

                entity_to_doc = defaultdict(list)

            

            print(f"✅ Loaded: {len(documents)} docs, {kg.number_of_nodes()} entities\n")

            

            return {

                'documents': documents,

                'titles': titles,

                'embeddings': embeddings,

                'kg': kg,

                'entity_to_doc': entity_to_doc

            }

        else:

            print("✅ No existing KG found - will create new one\n")

            return {

                'documents': [],

                'titles': [],

                'embeddings': None,

                'kg': nx.Graph(),

                'entity_to_doc': defaultdict(list)

            }



    def fetch_new_chunks(self, articles_to_scan=500, similarity_threshold=0.30):

        """Fetch new chunks with strict deduplication"""

        print(f"🔍 Scanning {articles_to_scan} new Wikipedia articles...")

        

        # Scientific query topics

        query_topics = [

            "physics quantum mechanics relativity electromagnetic",

            "chemistry periodic table molecular structure reactions",

            "biology cell DNA RNA evolution genetics",

            "mathematics calculus algebra geometry number theory",

            "computer science algorithms machine learning AI neural networks",

            "photosynthesis chloroplast plant enzyme protein"

        ]

        

        # Encode queries

        query_embeddings = self.embedder.encode(

            query_topics, 

            show_progress_bar=False, 

            convert_to_numpy=True

        )

        query_embeddings = query_embeddings / np.linalg.norm(query_embeddings, axis=1, keepdims=True)

        

        # Stream Wikipedia

        dataset = load_dataset(

            "wikimedia/wikipedia", 

            "20231101.en", 

            split="train", 

            streaming=True

        )

        

        candidates = []

        skipped = self.metadata['total_articles']

        new_processed = 0

        duplicates = 0

        

        print(f"⏭️ Skipping {skipped} already-processed articles...")

        

        for idx, item in enumerate(dataset):

            # Skip already processed

            if idx < skipped:

                continue

            

            if new_processed >= articles_to_scan:

                break

            

            title = item['title']

            text = item.get('text', '')

            

            if len(text) < 200:

                continue

            

            chunks = chunk_wikipedia_article(title, text)

            new_processed += 1

            

            for chunk in chunks:

                chunk_hash = compute_content_hash(chunk)

                

                # Check duplicates

                if chunk_hash in self.metadata['content_hashes']:

                    duplicates += 1

                    continue

                

                candidates.append({

                    'title': title,

                    'text': chunk,

                    'hash': chunk_hash

                })

            

            # Progress update every 100 articles

            if new_processed % 100 == 0:

                print(f"  Progress: {new_processed}/{articles_to_scan} articles "

                      f"→ {len(candidates)} unique chunks")

        

        print(f"✅ Found {len(candidates)} unique chunks ({duplicates} duplicates filtered)\n")

        

        if not candidates:

            print("⚠️ No new chunks found!")

            return [], [], []

        

        # Embed and filter by relevance

        texts = [c['text'] for c in candidates]

        print(f"⚡ Embedding {len(texts)} candidate chunks...")

        

        embeddings = self.embedder.encode(

            texts, 

            show_progress_bar=True, 

            batch_size=128

        )

        embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

        

        # Calculate relevance

        similarities = np.dot(embeddings_norm, query_embeddings.T)

        max_similarities = np.max(similarities, axis=1)

        

        filtered_indices = np.where(max_similarities >= similarity_threshold)[0]

        print(f"✅ {len(filtered_indices)} chunks above threshold {similarity_threshold}\n")

        

        if len(filtered_indices) == 0:

            return [], [], []

        

        new_docs = [candidates[i]['text'] for i in filtered_indices]

        new_titles = [candidates[i]['title'] for i in filtered_indices]

        new_hashes = [candidates[i]['hash'] for i in filtered_indices]

        

        # Update metadata

        self.metadata['total_articles'] += new_processed

        self.metadata['total_chunks'] += len(new_docs)

        self.metadata['content_hashes'].update(new_hashes)

        

        return new_docs, new_titles, new_hashes



    def extract_entities(self, text):

        """Fast entity extraction"""

        entities = set()

        

        if self.nlp:

            doc = self.nlp(text)

            for ent in doc.ents:

                if ent.label_ in ['PERSON', 'ORG', 'GPE', 'PRODUCT', 'EVENT']:

                    ent_clean = ent.text.strip().lower()

                    if 2 < len(ent_clean) < 40 and ent_clean not in self.stop_words:

                        entities.add(ent_clean)

        else:

            # Fallback: capitalized words

            words = text.split()

            for word in words:

                word = word.strip('.,;:!?()"\'')

                if word and len(word) > 2 and word[0].isupper():

                    entities.add(word.lower())

        

        return entities



    def merge_into_kg(self, existing_data, new_docs, new_titles):

        """Merge new documents into existing KG"""

        print(f"🔗 Merging {len(new_docs)} chunks into KG...")

        

        # Combine documents

        all_docs = existing_data['documents'] + new_docs

        all_titles = existing_data['titles'] + new_titles

        

        # Embed new docs

        print("⚡ Embedding new documents...")

        new_embeddings = self.embedder.encode(

            new_docs, 

            show_progress_bar=True, 

            batch_size=64

        )

        

        # Combine embeddings

        if existing_data['embeddings'] is not None:

            all_embeddings = np.vstack([existing_data['embeddings'], new_embeddings])

        else:

            all_embeddings = new_embeddings

        

        # Update Knowledge Graph

        kg = existing_data['kg']

        entity_to_doc = existing_data['entity_to_doc']

        start_idx = len(existing_data['documents'])

        

        print("🧠 Extracting entities and building relationships...")

        

        new_entities = 0

        for i, doc in enumerate(new_docs):

            doc_idx = start_idx + i

            

            entities = self.extract_entities(doc[:400])

            

            for ent in entities:

                entity_to_doc[ent].append(doc_idx)

                

                if ent not in kg:

                    kg.add_node(ent, count=1)

                    new_entities += 1

                else:

                    kg.nodes[ent]['count'] += 1

            

            # Co-occurrence edges

            ent_list = list(entities)[:10]

            for j, e1 in enumerate(ent_list):

                for e2 in ent_list[j+1:]:

                    if kg.has_edge(e1, e2):

                        kg[e1][e2]["weight"] += 1

                    else:

                        kg.add_edge(e1, e2, weight=1)

        

        print(f"✅ Added {new_entities} new entities")

        print(f"   Total: {kg.number_of_nodes()} entities, {kg.number_of_edges()} relations\n")

        

        return {

            'documents': all_docs,

            'titles': all_titles,

            'embeddings': all_embeddings,

            'kg': kg,

            'entity_to_doc': entity_to_doc

        }



    def rebuild_indices(self, data):

        """Rebuild FAISS and community indices"""

        print("🔄 Rebuilding search indices...")

        

        # FAISS semantic search

        dim = data['embeddings'].shape[1]

        neighbor_index = faiss.IndexFlatIP(dim)

        norm_embs = data['embeddings'].astype(np.float32)

        faiss.normalize_L2(norm_embs)

        neighbor_index.add(norm_embs)

        

        # Community detection

        n_docs = len(data['documents'])

        n_clusters = min(max(20, n_docs // 100), 100)

        print(f"⚡ Creating {n_clusters} communities for {n_docs} documents...")

        

        kmeans = KMeans(

            n_clusters=n_clusters, 

            random_state=42, 

            n_init=5, 

            max_iter=100

        )

        communities = kmeans.fit_predict(data['embeddings'])

        

        # Community centroids

        centroids = []

        for i in range(n_clusters):

            mask = (communities == i)

            if np.any(mask):

                centroid = np.mean(data['embeddings'][mask], axis=0)

                centroids.append(centroid)

        

        community_index = faiss.IndexFlatIP(dim)

        centroids_array = np.array(centroids).astype(np.float32)

        faiss.normalize_L2(centroids_array)

        community_index.add(centroids_array)

        

        # Generate community summaries

        print("📝 Generating community summaries...")

        tokenizer = AutoTokenizer.from_pretrained(f'{MODEL_DIR}/flan-t5-base')

        generator = AutoModelForSeq2SeqLM.from_pretrained(

            f"{MODEL_DIR}/flan-t5-base"

        ).to(self.device)

        

        community_summaries = {}

        for cid in set(communities):

            indices = [i for i, c in enumerate(communities) if c == cid]

            if not indices:

                continue

            

            # Get representative docs

            embeds = data['embeddings'][indices]

            centroid = np.mean(embeds, axis=0)

            sims = np.dot(embeds, centroid)

            top_k = min(2, len(indices))

            top_indices = np.argsort(sims)[-top_k:][::-1]

            top_chunks = [data['documents'][indices[i]][:400] for i in top_indices]

            

            prompt = "Summarize the main scientific themes:\n" + "\n".join(top_chunks)

            inputs = tokenizer(

                prompt, 

                return_tensors="pt", 

                max_length=512, 

                truncation=True

            )

            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            

            with torch.no_grad():

                out = generator.generate(

                    **inputs, 

                    max_length=120, 

                    num_beams=2, 

                    early_stopping=True

                )

            

            summary = tokenizer.decode(out[0], skip_special_tokens=True)

            community_summaries[cid] = summary

        

        print(f"✅ Generated {len(community_summaries)} community summaries\n")

        

        return {

            'neighbor_index': neighbor_index,

            'communities': communities,

            'community_index': community_index,

            'community_summaries': community_summaries

        }



    def save_merged_kg(self, data, indices):

        """Save merged KG to disk"""

        print("💾 Saving merged KG...")

        

        # Save documents and embeddings

        with open(f"{self.kg_db_path}/documents.pkl", "wb") as f:

            pickle.dump(data['documents'], f)

        with open(f"{self.kg_db_path}/titles.pkl", "wb") as f:

            pickle.dump(data['titles'], f)

        np.save(f"{self.kg_db_path}/document_embeddings.npy", data['embeddings'])

        

        # Save communities

        np.save(f"{self.kg_db_path}/communities.npy", np.array(indices['communities']))

        with open(f"{self.kg_db_path}/community_summaries.pkl", "wb") as f:

            pickle.dump(indices['community_summaries'], f)

        

        # Save knowledge graph

        with open(f"{self.kg_db_path}/knowledge_graph.pkl", "wb") as f:

            pickle.dump(data['kg'], f)

        with open(f"{self.kg_db_path}/entity_to_doc.pkl", "wb") as f:

            pickle.dump(dict(data['entity_to_doc']), f)

        

        # Update metadata

        self.metadata['last_update'] = datetime.now().isoformat()

        self.metadata['update_history'].append({

            'timestamp': datetime.now().isoformat(),

            'total_docs': len(data['documents']),

            'total_entities': data['kg'].number_of_nodes(),

            'total_relations': data['kg'].number_of_edges()

        })

        self._save_metadata()

        

        print("✅ All data saved successfully!\n")

        

        # Print summary

        print("="*60)

        print("📊 FINAL KG STATISTICS")

        print("="*60)

        print(f"Total Documents:    {len(data['documents']):,}")

        print(f"Total Entities:     {data['kg'].number_of_nodes():,}")

        print(f"Total Relations:    {data['kg'].number_of_edges():,}")

        print(f"Communities:        {len(set(indices['communities']))}")

        print("="*60)



    def incremental_update(self, articles_to_add=500, similarity_threshold=0.30):

        """Main method: Add new articles incrementally"""

        start_time = datetime.now()

        

        print("\n" + "="*60)

        print(f"🚀 INCREMENTAL UPDATE - Adding {articles_to_add} articles")

        print("="*60 + "\n")

        

        # Step 1: Load existing KG

        existing_data = self.load_existing_kg()

        

        # Step 2: Fetch new chunks

        new_docs, new_titles, new_hashes = self.fetch_new_chunks(

            articles_to_scan=articles_to_add,

            similarity_threshold=similarity_threshold

        )

        

        if not new_docs:

            print("\n⚠️ No new documents to add. Try again or lower threshold.")

            return

        

        # Step 3: Merge into KG

        merged_data = self.merge_into_kg(existing_data, new_docs, new_titles)

        

        # Step 4: Rebuild indices

        indices = self.rebuild_indices(merged_data)

        

        # Step 5: Save

        self.save_merged_kg(merged_data, indices)

        

        # Summary

        elapsed = (datetime.now() - start_time).total_seconds() / 60

        print(f"\n⏱️ Update completed in {elapsed:.1f} minutes")

        print(f"✅ Next run will start from article {self.metadata['total_articles']}")

        

        # Show history

        if len(self.metadata['update_history']) > 1:

            print(f"\n📖 Update History ({len(self.metadata['update_history'])} runs):")

            for i, update in enumerate(self.metadata['update_history'][-3:], 1):

                print(f"   Run {i}: {update['total_docs']} docs, "

                      f"{update['total_entities']} entities")



def main():

    """

    USAGE GUIDE:

    ------------

    Run this script multiple times to grow your KG incrementally.

    Each run adds ~500 articles (usually 200-400 relevant chunks).

    

    Example progression:

    - Run 1:  500 articles → ~300 chunks   (Build initial KG)

    - Run 2:  500 more    → ~600 total     (Add more content)

    - Run 3:  500 more    → ~900 total     (Continue growing)

    - Run 4:  500 more    → ~1200 total

    - ... and so on ...

    

    Each run takes ~5-10 minutes on Kaggle GPU.

    """

    

    loader = IncrementalKGLoader()

    

    # Small batch size for frequent updates

    ARTICLES_PER_RUN = 500

    

    loader.incremental_update(articles_to_add=ARTICLES_PER_RUN)

    

    print("\n" + "="*60)

    print("💡 TIP: Run this script again to add another 500 articles!")

    print("="*60 + "\n")



if __name__ == "__main__":

    main()

🚀 Incremental KG Loader initialized on cuda
📊 Current state:
   Articles processed: 1000
   Chunks stored: 680
   Last update: 2025-12-02T20:18:08.391939
⚡ Loading models...
✅ Models loaded


🚀 INCREMENTAL UPDATE - Adding 500 articles

📂 Loading existing KG...
✅ Loaded: 680 docs, 1009 entities

🔍 Scanning 500 new Wikipedia articles...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

⏭️ Skipping 1000 already-processed articles...
  Progress: 100/500 articles → 936 unique chunks
  Progress: 200/500 articles → 1860 unique chunks
  Progress: 300/500 articles → 3070 unique chunks
  Progress: 400/500 articles → 4089 unique chunks
  Progress: 500/500 articles → 6008 unique chunks
✅ Found 6008 unique chunks (0 duplicates filtered)

⚡ Embedding 6008 candidate chunks...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

✅ 184 chunks above threshold 0.3

🔗 Merging 184 chunks into KG...
⚡ Embedding new documents...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

🧠 Extracting entities and building relationships...
✅ Added 212 new entities
   Total: 1221 entities, 2832 relations

🔄 Rebuilding search indices...
⚡ Creating 20 communities for 864 documents...
📝 Generating community summaries...
✅ Generated 20 community summaries

💾 Saving merged KG...
✅ All data saved successfully!

📊 FINAL KG STATISTICS
Total Documents:    864
Total Entities:     1,221
Total Relations:    2,832
Communities:        20

⏱️ Update completed in 3.1 minutes
✅ Next run will start from article 1500

📖 Update History (3 runs):
   Run 1: 482 docs, 646 entities
   Run 2: 680 docs, 1009 entities
   Run 3: 864 docs, 1221 entities

💡 TIP: Run this script again to add another 500 articles!



In [45]:
from kaggle_persistence import save_kg_to_output
save_kg_to_output() 

  ✅ documents.pkl (42.33 MB)
  ✅ titles.pkl (0.03 MB)
  ✅ document_embeddings.npy (5.86 MB)
  ✅ knowledge_graph.pkl (0.64 MB)
  ✅ entity_contexts.pkl (0.91 MB)
  ✅ entity_cooccurrence.pkl (0.36 MB)
  ✅ community_summaries.pkl (0.01 MB)
  ✅ communities.npy (0.01 MB)
  ✅ metadata.json (0.00 MB)
  ✅ query_history.json (0.00 MB)

✅ Saved 10 files to output


In [46]:
#-------------------------------------------------------#

# Create .tar.gz for Kaggle dataset upload (same as finalLightRAG)

import shutil, os

shutil.make_archive('/kaggle/working/hybrid_kg_dataset', 'gztar', '/kaggle/working/kg_database')

print("✅ Archive ready for Kaggle dataset upload")

✅ Archive ready for Kaggle dataset upload
